In [ ]:
pip install langgraph langchain langchain-community groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 4.0 MB/s eta 0:00:00


In [ ]:
import os
os.environ["GROQ_API_KEY"] = "YOUR_API_KEY"

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph
from groq import Groq

class AgentState(TypedDict):
    user_input: str
    decision: str
    response: str


In [ ]:
client = Groq(api_key=os.environ["GROQ_API_KEY"])

In [ ]:
def input_node(state: AgentState):
    return state

def router_node(state: AgentState):
    if "search" in state["user_input"].lower():
        return "tool"
    else:
        return "llm"



In [ ]:
def tool_node(state: AgentState):
    query = state["user_input"]
    return {"response": f"Search results for: {query}"}

def llm_node(state: AgentState):
    response = client.chat.completions.create(
        model="mixtral-8x7b-32768", # Groq model
        messages=[{"role": "user", "content": state["user_input"]}]
    )
    answer = response.choices[0].message.content
    return {"response": answer}


In [ ]:
def final_node(state: AgentState):
    return {"response": state.get("response", "No response")}

graph = StateGraph(AgentState)
graph.add_node("input", input_node)
graph.add_node("router", router_node)
graph.add_node("tool", tool_node)
graph.add_node("llm", llm_node)
graph.add_node("final", final_node)

graph.set_entry_point("input")
graph.add_edge("input", "router")
graph.add_conditional_edges("router", router_node, {"tool": "tool", "llm": "llm"})
graph.add_edge("tool", "final")
graph.add_edge("llm", "final")

app = graph.compile()


In [ ]:
print(app.invoke({"user_input": "search LangGraph"}))
print(app.invoke({"user_input": "Explain LangGraph"}))


InvalidUpdateError: Expected dict, got tool
For troubleshooting, visit: https://docs.langchain.com/oss/python/langgraph/errors/INVALID_GRAPH_NODE_RETURN_VALUE